# Task 6 · Hugging Face → Drive
Clone the shared semeval repository and install the package in `task 6/`.
No code ZIP or local dataset upload. Edit REPO_URL / GIT_REF in the cell below.
Use Runtime → Change runtime type → GPU for training. The transfer notebook can run on CPU.


In [ ]:
# Repository settings: editable in the notebook.
REPO_URL = 'https://github.com/iamtro-2006/semeval.git'
GIT_REF = 'main'  # branch, tag, or commit
CHECKOUT = '/content/semeval'
PROJECT_SUBDIR = 'task 6'

from pathlib import Path
import os, subprocess, sys
checkout = Path(CHECKOUT)
if not checkout.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(checkout)], check=True)
else:
    actual = subprocess.check_output(['git', '-C', str(checkout), 'remote', 'get-url', 'origin'], text=True).strip()
    if actual != REPO_URL:
        raise ValueError('CHECKOUT belongs to a different repository; choose another CHECKOUT')
    dirty = subprocess.check_output(['git', '-C', str(checkout), 'status', '--porcelain'], text=True)
    if dirty.strip():
        raise RuntimeError('Checkout has edits; preserve your changes or choose a fresh CHECKOUT before switching revision')
subprocess.run(['git', '-C', str(checkout), 'fetch', 'origin', GIT_REF], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
os.chdir(checkout / PROJECT_SUBDIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation', '-e', '.'], check=True)
print('Project:', Path.cwd())
print('Commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())


## MMCQA folder and authentication
Only ROOT_FOLDER_ID selects storage. Paste the ID or URL of your MMCQA folder.
RCLONE_CONFIG_PATH points to your authenticated rclone.conf; this is not a storage root.
All data, adapters, checkpoints and results are addressed relative to the selected ID.
If your config is on mounted Drive, run the optional mount cell first.


In [ ]:
# Optional, only to access a config file stored on Drive:
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
ROOT_FOLDER_ID = '1kcdByZiSJzGSoEtY9q0OEhsZwiTZkxpV'  # ID or URL of the MMCQA folder
RCLONE_CONFIG_PATH = '/content/rclone.conf'
REMOTE = 'iamtro'  # Google Drive remote in rclone.conf


In [ ]:
!sudo apt-get -qq update && sudo apt-get -qq install -y rclone
import configparser, re, tempfile
from urllib.parse import urlparse
config_source = Path(RCLONE_CONFIG_PATH).expanduser()
if not config_source.is_file():
    raise FileNotFoundError(f'Upload your authenticated rclone.conf or mount Drive: {config_source}')
sections = configparser.RawConfigParser()
sections.read(config_source, encoding='utf-8-sig')
remotes = [name for name in sections.sections() if sections[name].get('type') == 'drive']
if not REMOTE:
    if len(remotes) != 1:
        raise ValueError(f'Select REMOTE from Google Drive sections: {remotes}')
    REMOTE = remotes[0]
if REMOTE not in remotes:
    raise ValueError('REMOTE must be a Google Drive section')
root_id = ROOT_FOLDER_ID.strip()
if root_id.startswith(('https://', 'http://')):
    match = re.search(r'/folders/([^/?]+)', urlparse(root_id).path)
    root_id = match.group(1) if match else ''
if not re.fullmatch(r'[A-Za-z0-9_-]+', root_id):
    raise ValueError('Enter the ID or folder URL of MMCQA in ROOT_FOLDER_ID')
temporary_config = tempfile.TemporaryDirectory(prefix='mmcqa-rclone-')
config_path = Path(temporary_config.name) / 'rclone.conf'
config_path.write_bytes(config_source.read_bytes())
config_path.chmod(0o600)
rclone_base = ['rclone', '--config', str(config_path), '--drive-root-folder-id', root_id]
def cloud(relative):
    if relative.startswith('/') or '..' in Path(relative).parts:
        raise ValueError('Storage paths must be relative to MMCQA')
    return f'{REMOTE}:{relative}'
def copy(source, destination, *extra):
    subprocess.run(rclone_base + ['copy', str(source), str(destination),
                   '--transfers', '4', '--checkers', '8', '--progress', *extra], check=True)
subprocess.run(rclone_base + ['lsd', cloud(''), '--max-depth', '1'], check=True)
print('All storage is relative to MMCQA folder ID:', root_id)


## Download English QA and upload automatically below MMCQA
Includes train/dev/devtest JSONL, Parquet and image ZIPs; excludes audio and other languages.
Only compressed archives and labels are uploaded. All later notebooks use the same relative data path.


In [ ]:
%pip -q install huggingface-hub==0.36.0 Pillow==11.3.0
!python -m mmcultureqa.data.prepare download
!python -m mmcultureqa.data.prepare prepare


In [ ]:
filters = ['--exclude', '.cache/**', '--exclude', 'train/**', '--exclude', 'dev/**', '--exclude', 'devtest/**']
copy('data/raw', cloud('task6-data/raw'), *filters)
subprocess.run(rclone_base + ['check', 'data/raw', cloud('task6-data/raw'),
                '--one-way', '--size-only', *filters], check=True)
print('Dataset ready under MMCQA/task6-data/raw')
